# InaTRC · Native KD YOLO26n

Notebook khusus **Native KD YOLO26n**, satu eksperimen per Run All. **Run All default hanya smoke sintetis nonfinal**: satu epoch, input kecil, model acak, tanpa mengunduh bobot pretrained. Bila dataset InaTRC terpasang, audit dilakukan sebelum smoke; hasil smoke bukan hasil skripsi.

Aktifkan GPU di Settings → Accelerator. Isi sumber repository sekali: URL GitHub + commit SHA lengkap, atau snapshot source di Kaggle Input. Algoritme dan protokol berada di package/config repository.
Notebook ini dibuat dari runner umum; setup memakai template yang sama dan training tetap di repository.


Stage, model, dan metode sudah tetap. Ubah seed, mode, sumber repository, dataset, checkpoint teacher, device, dan W&B sesuai kebutuhan. Default **smoke, seed 42, W&B disabled**.

In [ ]:
# Pilihan eksperimen — hanya satu run per Run All
STAGE = "student"  # preset tetap; gunakan notebook lain untuk eksperimen berbeda
STUDENT = "n"  # preset model notebook ini
METHOD = "native"  # preset metode notebook ini
SEED = 42
MODE = "smoke"             # smoke / full; full harus dipilih eksplisit
DATASET_ROOT = ""          # kosong: deteksi root unik pada INPUT_ROOT
TEACHER_CKPT = ""          # checkpoint teacher tetap; wajib untuk KD full
MODEL_WEIGHTS = None      # opsional: path pretrained COCO .pt untuk full/offline
DEVICE = "0"
WANDB_MODE = "disabled"    # disabled / offline / online
WANDB_PROJECT = "inatrc-yolo26-kd"
WANDB_ENTITY = ""          # opsional, identitas milik Anda

# Sumber kode dan setup — isi salah satu REPO_URL atau SNAPSHOT_ROOT
REPO_URL = ""              # contoh: https://github.com/USER/REPO.git
REPO_COMMIT = ""           # wajib SHA commit lengkap 40 karakter untuk clone
SNAPSHOT_ROOT = ""         # root snapshot yang berisi pyproject.toml dan uv.lock
OFFLINE = False
UV_BINARY = ""             # binary uv Linux 0.9.8 dari Input untuk offline
UV_CACHE_INPUT = ""        # cache uv Linux sesuai Python kernel, termasuk build backend

PLATFORM = "kaggle"
WORK_ROOT = "/kaggle/working"
INPUT_ROOT = "/kaggle/input"
OUTPUT_ROOT = "/kaggle/working/outputs"

# COLAB PATHS BEGIN — uncomment empat assignment ini saat memakai Colab.
# PLATFORM = "colab"
# WORK_ROOT = "/content"
# INPUT_ROOT = "/content/inputs"
# OUTPUT_ROOT = "/content/outputs"
# COLAB PATHS END
# Upload dataset/snapshot ke INPUT_ROOT atau isi DATASET_ROOT/SNAPSHOT_ROOT.
# Clone GitHub tetap memakai REPO_URL + REPO_COMMIT di atas.

# COLAB DRIVE BEGIN — opsional; uncomment hanya jika input ada di Drive.
# from google.colab import drive
# drive.mount("/content/drive")
# INPUT_ROOT = "/content/drive/MyDrive/inatrc-inputs"
# COLAB DRIVE END
# Output tetap lokal /content/outputs; unduh/salin hasil ke Drive sebelum sesi berakhir.

# Cegah eksperimen berubah tanpa sengaja; pilih file notebook yang sesuai.
assert (STAGE, STUDENT, METHOD) == ('student', 'n', 'native'), (
    "Preset 05_native_yolo26n harus tetap: stage=student, student=n, method=native. Pilih notebook lain untuk eksperimen berbeda."
)


## 1. Siapkan source yang dapat dilacak

Ambil hanya commit pilihan (depth 1, tanpa history/tag lain). Folder source yang sudah sesuai dipakai ulang. Snapshot disalin ke WORK_ROOT; fingerprint source dicatat. Input hanya dibaca. Snapshot offline perlu dependency cache dan uv Linux yang sudah disiapkan; source saja belum cukup.

**Opsional Colab:** di cell konfigurasi, uncomment empat assignment pada `COLAB PATHS`. Pilih Runtime → Change runtime type → GPU. Isi sumber kode seperti pada Kaggle; taruh input di `/content/inputs` atau gunakan blok `COLAB DRIVE` opsional. Jika W&B online, uncomment dua baris `COLAB SECRET` pada cell Secrets dan aktifkan Notebook access untuk secret `WANDB_API_KEY`. Setup uv bersama di bawah otomatis memakai path Colab; tidak perlu menyalin cell instalasi. Kernel wajib Python 3.11/3.12/3.13 dan Torch/Torchvision kompatibel; Python 3.13 memerlukan Torch ≥2.7/torchvision ≥0.22. Runtime GPU Colab belum diuji.
Tahap source menampilkan durasi dan menyimpan log di `OUTPUT_ROOT/setup_logs/source.log`. Fetch terputus dapat dilanjutkan bila checkout masih kosong. Source berbeda atau perubahan pengguna dihentikan dengan pesan jelas.


In [ ]:
from contextlib import contextmanager, redirect_stdout, redirect_stderr
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import time
from urllib.parse import urlsplit

# Log setup dapat diunduh meskipun training belum sempat dimulai.
LOG_DIR = Path(OUTPUT_ROOT) / "setup_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
BOOTSTRAP_READY = False
_active_log = None

def redact(text):
    # Key hanya dibaca untuk penyamaran; nilainya tidak pernah ditampilkan.
    for environment in (os.environ, globals().get("CHILD_ENV", {})):
        secret = environment.get("WANDB_API_KEY")
        if secret:
            text = text.replace(secret, "[REDACTED]")
    return text

class SetupOutput:
    def __init__(self, screen, log):
        self.screen, self.log = screen, log

    def write(self, text):
        clean = redact(text)
        self.screen.write(clean)
        self.log.write(clean)
        self.flush()
        return len(text)

    def flush(self):
        self.screen.flush()
        self.log.flush()

    def isatty(self):
        return False

@contextmanager
def setup_stage(name):
    global _active_log, BOOTSTRAP_READY
    BOOTSTRAP_READY = False
    _active_log = LOG_DIR / (name + ".log")
    started = time.perf_counter()
    with _active_log.open("a", encoding="utf-8") as log:
        with redirect_stdout(SetupOutput(sys.stdout, log)), redirect_stderr(SetupOutput(sys.stderr, log)):
            print(f"[{name}] Mulai {datetime.now(timezone.utc).isoformat()} | log: {_active_log}", flush=True)
            try:
                yield
            except (Exception, KeyboardInterrupt) as error:
                message = redact(str(error)) or type(error).__name__
                print(f"[{name}] GAGAL setelah {time.perf_counter() - started:.1f} detik: {message}", flush=True)
                raise RuntimeError(f"Tahap {name} gagal: {message[:1000]}. Log lengkap: {_active_log}") from None
            else:
                print(f"[{name}] Selesai: {time.perf_counter() - started:.1f} detik", flush=True)

def run_logged(command, *, cwd=None, env=None, check=True, capture=False):
    # Download tetap menampilkan progres; stdout JSON bisa ditangkap untuk pemeriksaan.
    output = []
    last_line = ""
    with subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace") as process:
        for line in process.stdout:
            clean = redact(line)
            if clean.strip():
                last_line = clean.strip()
            if capture:
                output.append(clean)
                with _active_log.open("a", encoding="utf-8") as log:
                    log.write(clean)
            else:
                print(clean, end="", flush=True)
        returncode = process.wait()
    result = subprocess.CompletedProcess(command, returncode, "".join(output))
    if check and returncode:
        raise RuntimeError(f"{Path(command[0]).name} gagal (exit {returncode}): {last_line[:300]}")
    return result

def prepare_git_source(repo, url, commit):
    # Jangan menimpa source pengguna; repo kosong boleh melanjutkan fetch terputus.
    if repo.exists() and not (repo / ".git").is_dir():
        raise RuntimeError("Folder Working bukan checkout Git; gunakan sesi baru atau path source yang sesuai.")
    if not repo.exists():
        run_logged(["git", "init", "--quiet", str(repo)])
    marker = repo / ".git" / "inatrc_source.json"
    expected = {"url": url, "commit": commit.lower()}
    if marker.is_file() and json.loads(marker.read_text(encoding="utf-8")) != expected:
        raise RuntimeError("Source Working berasal dari URL/commit berbeda; gunakan sesi baru.")
    head = run_logged(["git", "rev-parse", "--verify", "HEAD"], cwd=repo, check=False, capture=True)
    if head.returncode:
        if any(path.name != ".git" for path in repo.iterdir()):
            raise RuntimeError("Checkout tanpa commit memiliki file pengguna; tidak dilanjutkan otomatis.")
        marker.write_text(json.dumps(expected), encoding="utf-8")
        run_logged(["git", "fetch", "--depth", "1", "--no-tags", "--progress", url, commit], cwd=repo)
        run_logged(["git", "checkout", "--detach", commit], cwd=repo)
        head = run_logged(["git", "rev-parse", "--verify", "HEAD"], cwd=repo, capture=True)
    actual_commit = head.stdout.strip()
    if actual_commit.lower() != commit.lower():
        raise RuntimeError("Source Working memiliki commit berbeda; gunakan sesi baru atau source yang sesuai.")
    dirty = run_logged(["git", "status", "--porcelain"], cwd=repo, capture=True)
    if dirty.stdout.strip():
        raise RuntimeError("Source Working memiliki perubahan/file pengguna; gunakan source bersih.")
    marker.write_text(json.dumps(expected), encoding="utf-8")
    print("Source Git siap:", actual_commit)
    return actual_commit

with setup_stage("source"):
    assert PLATFORM in {"kaggle", "colab"}
    assert Path(WORK_ROOT).is_dir(), "WORK_ROOT tidak ditemukan; pilih blok path platform yang sesuai."
    assert (3, 11) <= sys.version_info[:2] < (3, 14), "Gunakan Python 3.11, 3.12, atau 3.13."
    assert STAGE in {"teacher", "student"}
    assert STUDENT in {"n", "s"}
    assert METHOD in {"none", "native", "crosskd", "csakd"}
    assert MODE in {"smoke", "full"}
    assert WANDB_MODE in {"disabled", "offline", "online"}
    assert STAGE != "teacher" or METHOD == "none", "Teacher menggunakan METHOD=none."
    assert not (OFFLINE and WANDB_MODE == "online"), "W&B online memerlukan internet."

    REPO = Path(WORK_ROOT) / "inatrc-kd-yolov26"
    INPUT = Path(INPUT_ROOT).resolve()
    if PLATFORM == "colab" and INPUT_ROOT == "/content/inputs":
        INPUT.mkdir(parents=True, exist_ok=True)
    if not INPUT.is_dir():
        raise FileNotFoundError("INPUT_ROOT tidak ditemukan; upload input atau periksa mount Drive/path.")
    SKIP_DIRS = {".git", ".venv", "__pycache__", ".ipynb_checkpoints", "outputs", "runs", "wandb", ".pytest_cache", ".uv-cache", ".cache"}

    def source_files(root):
        # Jangan masuk ke venv/cache; mengabaikan setelah rglob tetap menelusuri ribuan file.
        for directory, folders, filenames in os.walk(root):
            folders[:] = sorted(folder for folder in folders if folder not in SKIP_DIRS)
            for filename in sorted(filenames):
                path = Path(directory) / filename
                if path.is_file() and (not path.name.startswith(".env") or path.name == ".env.example") and path.suffix != ".pt":
                    yield path, path.relative_to(root)

    def source_digest(root):
        digest = hashlib.sha256()
        for path, relative in source_files(root):
            digest.update(relative.as_posix().encode() + b"\0")
            digest.update(hashlib.sha256(path.read_bytes()).digest())
        return digest.hexdigest()

    # Commit/fingerprint mencatat source persis yang dipakai oleh eksperimen.
    print("Source: mengambil commit/snapshot atau memakai folder yang sudah sesuai...", flush=True)
    if REPO_URL:
        parsed = urlsplit(REPO_URL)
        if parsed.scheme != "https" or not parsed.hostname or parsed.username or parsed.password or parsed.query or parsed.fragment:
            raise ValueError("Gunakan URL HTTPS repository publik tanpa credential, query, atau fragment.")
        if SNAPSHOT_ROOT or OFFLINE:
            raise ValueError("Clone membutuhkan internet dan tidak boleh digabung dengan snapshot.")
        if not re.fullmatch(r"[0-9a-fA-F]{40}", REPO_COMMIT):
            raise ValueError("REPO_COMMIT harus SHA commit lengkap 40 karakter.")
        actual_commit = prepare_git_source(REPO, REPO_URL, REPO_COMMIT)
        provenance = {"kind": "git", "commit": actual_commit, "source_sha256": source_digest(REPO)}
    else:
        if SNAPSHOT_ROOT:
            snapshot = Path(SNAPSHOT_ROOT).resolve()
        else:
            candidates = sorted({p.parent.resolve() for p in INPUT.rglob("pyproject.toml") if (p.parent / "uv.lock").is_file()})
            if len(candidates) != 1:
                raise ValueError("Isi SNAPSHOT_ROOT atau REPO_URL+REPO_COMMIT. Kandidat snapshot: " + str(candidates))
            snapshot = candidates[0]
        if snapshot != INPUT and INPUT not in snapshot.parents:
            raise ValueError("SNAPSHOT_ROOT harus berada di INPUT_ROOT.")
        if not (snapshot / "pyproject.toml").is_file() or not (snapshot / "uv.lock").is_file():
            raise ValueError("Snapshot harus memiliki pyproject.toml dan uv.lock.")
        fingerprint = source_digest(snapshot)
        if REPO.exists() and source_digest(REPO) != fingerprint:
            raise RuntimeError("Source Working berbeda dari snapshot; gunakan sesi baru.")
        REPO.mkdir(parents=True, exist_ok=True)
        for path, relative in source_files(snapshot):
            target = REPO / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
        provenance = {"kind": "snapshot", "commit": None, "source_path": str(snapshot), "source_sha256": fingerprint}

    provenance["uv_lock_sha256"] = hashlib.sha256((REPO / "uv.lock").read_bytes()).hexdigest()
    PROVENANCE_PATH = Path(WORK_ROOT) / "source_provenance.json"
    PROVENANCE_PATH.write_text(json.dumps(provenance, indent=2), encoding="utf-8")
    print(json.dumps(provenance, indent=2))


## 2. Gunakan uv tanpa mengganti Torch CUDA kernel

Bootstrap pip hanya memasang **uv 0.9.8**. Environment project mewarisi Torch/Torchvision kernel, sedangkan dependency lain mengikuti lock. Semua proses package memakai interpreter venv melalui `uv run --no-sync`; kernel notebook bukan interpreter training.

Versi CUDA Torch preinstalled merupakan pengecualian lock yang dicatat. Kecocokan GPU diperiksa, belum dijamin hanya oleh lockfile.

Setup pertama pada sesi baru perlu download paket sesuai lock. Run All ulang pada sesi yang sama memakai venv/cache yang sudah tersedia; paket yang sudah sesuai tidak dipasang ulang. Jangan restart sesi di antara smoke metode yang berbeda kecuali ada masalah environment. Dependency test tidak dipasang (`--no-dev`). Waktu setup dicetak agar download dependency dapat dibedakan dari training.
Download dependency menampilkan progres; log lengkap ada di `OUTPUT_ROOT/setup_logs/dependencies.log`. Venv/cache pada Working dipakai ulang dalam sesi yang sama.


In [ ]:
with setup_stage("dependencies"):
    print("Setup uv/venv: Torch CUDA dipertahankan, dependency lain mengikuti lock...", flush=True)
    import torch as kernel_torch
    import torchvision as kernel_vision
    KERNEL_TORCH = {"torch_file": str(Path(kernel_torch.__file__).resolve()), "torch_version": kernel_torch.__version__,
                    "torchvision_file": str(Path(kernel_vision.__file__).resolve()), "torchvision_version": kernel_vision.__version__,
                    "python": sys.version.split()[0]}

    if UV_BINARY:
        binary_input = Path(UV_BINARY).resolve()
        if not binary_input.is_file():
            raise FileNotFoundError("UV_BINARY tidak ditemukan.")
        binary_dir = Path(WORK_ROOT) / "uv-bin"
        binary_dir.mkdir(exist_ok=True)
        uv_path = binary_dir / "uv"
        if binary_input != uv_path.resolve():
            shutil.copy2(binary_input, uv_path)
        uv_path.chmod(0o755)
        UV = [str(uv_path)]
    else:
        # Module milik interpreter kernel mencegah uv lain di PATH terpilih.
        UV = [sys.executable, "-m", "uv"]
        found = run_logged(UV + ["--version"], check=False, capture=True)
        if found.returncode or not found.stdout.strip().startswith("uv 0.9.8 "):
            if OFFLINE:
                raise RuntimeError("Offline memerlukan binary uv Linux 0.9.8 melalui UV_BINARY atau module kernel yang sesuai.")
            run_logged([sys.executable, "-m", "pip", "install", "--quiet", "uv==0.9.8"])
    uv_version = run_logged(UV + ["--version"], capture=True).stdout.strip()
    if not uv_version.startswith("uv 0.9.8 "):
        raise RuntimeError("Runner ini diuji dengan uv 0.9.8; versi yang dipilih: " + uv_version)

    CHILD_ENV = os.environ.copy()
    CHILD_ENV["UV_PYTHON_DOWNLOADS"] = "never"
    CHILD_ENV["UV_PROJECT_ENVIRONMENT"] = str(REPO / ".venv")
    CHILD_ENV["UV_CACHE_DIR"] = str(Path(WORK_ROOT) / "uv-cache")
    CHILD_ENV["INATRC_SOURCE_COMMIT"] = provenance["commit"] or ""
    CHILD_ENV["INATRC_SOURCE_FINGERPRINT"] = provenance["source_sha256"]
    CHILD_ENV["INATRC_SOURCE_PROVENANCE"] = str(PROVENANCE_PATH)
    CHILD_ENV["WANDB_MODE"] = WANDB_MODE
    CHILD_ENV["WANDB_PROJECT"] = WANDB_PROJECT
    if WANDB_ENTITY:
        CHILD_ENV["WANDB_ENTITY"] = WANDB_ENTITY
    if OFFLINE:
        if not UV_CACHE_INPUT or not Path(UV_CACHE_INPUT).is_dir():
            raise RuntimeError("Offline memerlukan UV_CACHE_INPUT yang telah diisi dependency + build backend Linux/Python kernel.")
        cache_source = Path(UV_CACHE_INPUT).resolve()
        cache_target = Path(CHILD_ENV["UV_CACHE_DIR"]).resolve()
        if cache_source != cache_target:
            shutil.copytree(cache_source, cache_target, dirs_exist_ok=True)
        CHILD_ENV["UV_OFFLINE"] = "1"

    # Venv untuk proses training; Torch CUDA tetap diwarisi dari kernel notebook.
    venv = REPO / ".venv"
    if not (venv / "pyvenv.cfg").exists():
        run_logged(UV + ["venv", "--python", sys.executable, "--system-site-packages", "--no-python-downloads", str(venv)], cwd=REPO, env=CHILD_ENV)
    else:
        venv_config = (venv / "pyvenv.cfg").read_text(encoding="utf-8").lower()
        if "include-system-site-packages = true" not in venv_config:
            raise RuntimeError("Venv yang ada tidak mewarisi system site-packages; gunakan sesi baru.")

    sync = UV + ["sync", "--frozen", "--no-dev", "--python", sys.executable, "--no-python-downloads", "--no-install-package", "torch", "--no-install-package", "torchvision"]
    if OFFLINE:
        sync.append("--offline")
    run_logged(sync, cwd=REPO, env=CHILD_ENV)
    print(uv_version)
    print("Kernel Torch dipertahankan:", KERNEL_TORCH)

    # Semua pemanggilan berikut wajib --no-sync agar Torch CPU dalam lock tidak dipasang.
    UV_RUN = UV + ["run", "--frozen", "--no-sync", "--python", sys.executable, "python", "-m", "inatrc_kd"]

## 3. Secrets, dataset, dan preflight

W&B optional. Untuk mode online, tambahkan secret bernama `WANDB_API_KEY` dan aktifkan akses notebook: Kaggle Add-ons → Secrets atau Colab panel 🔑 Secrets → Notebook access. Di Colab uncomment dua baris bertanda `COLAB SECRET` di bawah. Environment yang sudah berisi key diprioritaskan. Secret hanya diteruskan melalui environment subprocess, tidak melalui argumen command atau output.

Dataset asli tidak dikonversi atau ditulis ulang. Audit memeriksa ketiga split; training/debugging hanya memakai train/val. Smoke tetap memakai data sintetis meskipun InaTRC diaudit.
Jika pemeriksaan gagal, training berhenti. Pesan error menyebut tahap dan path log `OUTPUT_ROOT/setup_logs/preflight.log`; unduh log itu dari panel Output untuk diagnosis. API key tidak dicetak atau ditulis ke log.


In [ ]:
with setup_stage("preflight"):
    if WANDB_MODE == "online" and not CHILD_ENV.get("WANDB_API_KEY"):
        try:
            if PLATFORM == "colab":
                # COLAB SECRET BEGIN — uncomment dua baris berikut untuk W&B online.
                # from google.colab import userdata
                # CHILD_ENV["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
                # COLAB SECRET END
                if not CHILD_ENV.get("WANDB_API_KEY"):
                    raise RuntimeError("Blok COLAB SECRET belum diaktifkan.")
            else:
                from kaggle_secrets import UserSecretsClient
                CHILD_ENV["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
            if not CHILD_ENV.get("WANDB_API_KEY"):
                raise RuntimeError("Secret kosong.")
        except Exception:
            raise RuntimeError("Aktifkan akses secret WANDB_API_KEY untuk W&B online; di Colab uncomment dua baris COLAB SECRET.") from None
    if WANDB_MODE != "online":
        CHILD_ENV.pop("WANDB_API_KEY", None)

    splits = ("train", "val", "test")
    def is_dataset_root(path):
        return all((path / split / "images").is_dir() and (path / split / "labels").is_dir() for split in splits)

    if DATASET_ROOT:
        data_root = Path(DATASET_ROOT).resolve()
        if not is_dataset_root(data_root):
            raise ValueError("DATASET_ROOT harus memiliki train/val/test, masing-masing images dan labels.")
    else:
        data_candidates = sorted({path.parent.parent.resolve() for path in INPUT.rglob("images") if path.parent.name == "train" and is_dataset_root(path.parent.parent)})
        if len(data_candidates) > 1:
            raise ValueError("Dataset ambigu; isi DATASET_ROOT. Kandidat: " + str(data_candidates))
        data_root = data_candidates[0] if data_candidates else None
    if MODE == "full" and data_root is None:
        raise ValueError("MODE=full memerlukan dataset InaTRC lengkap.")
    if MODE == "full" and METHOD != "none" and not TEACHER_CKPT:
        raise ValueError("KD full memerlukan checkpoint teacher yang tetap.")
    if MODEL_WEIGHTS:
        if MODE != "full":
            raise ValueError("MODEL_WEIGHTS hanya dipakai untuk MODE=full; smoke memakai inisialisasi acak.")
        if not Path(MODEL_WEIGHTS).is_file():
            raise FileNotFoundError("MODEL_WEIGHTS tidak ditemukan.")
    if OFFLINE and MODE == "full" and not MODEL_WEIGHTS:
        raise ValueError("Full offline memerlukan MODEL_WEIGHTS lokal pretrained COCO yang sesuai model.")
    if TEACHER_CKPT and not Path(TEACHER_CKPT).is_file():
        raise FileNotFoundError("TEACHER_CKPT tidak ditemukan.")

    checked = run_logged(UV_RUN + ["preflight", "--device", str(DEVICE), "--require-gpu", "--output-root", OUTPUT_ROOT], cwd=REPO, env=CHILD_ENV, capture=True)
    preflight = json.loads(checked.stdout)
    active_torch = preflight["environment"]
    if str(Path(active_torch["torch_file"]).resolve()) != KERNEL_TORCH["torch_file"] or active_torch["torch_version"] != KERNEL_TORCH["torch_version"]:
        raise RuntimeError("Torch environment project berbeda dari Torch kernel; hentikan dan periksa setup.")
    if str(Path(active_torch["torchvision_file"]).resolve()) != KERNEL_TORCH["torchvision_file"] or active_torch["torchvision_version"] != KERNEL_TORCH["torchvision_version"]:
        raise RuntimeError("Torchvision environment project berbeda dari kernel; hentikan dan periksa setup.")
    if active_torch["python"] != KERNEL_TORCH["python"]:
        raise RuntimeError("Versi Python venv berbeda dari kernel; jangan mencampur ABI interpreter.")
    print(json.dumps(preflight, indent=2))
    print("Dataset audit:", str(data_root) if data_root else "Tidak ada input InaTRC; smoke sintetis saja.")
BOOTSTRAP_READY = True


## 4. Jalankan satu eksperimen

`none` menjalankan baseline/teacher; `native` memakai API resmi Ultralytics. CrossKD dan CSAKD memakai adaptasi Tahap B di `docs/stage_b_design.md`; smoke tersedia untuk diagnosis. Semua KD full memerlukan teacher InaTRC tetap lima kelas. KD smoke bukan bukti kualitas metode. Jangan memilih `full` sebelum protokol final disetujui.

In [ ]:
# Training hanya boleh dimulai setelah seluruh setup berhasil.
assert BOOTSTRAP_READY, "Jalankan ulang source, setup uv, dan preflight yang gagal sebelum training."
# Notebook menyerahkan pilihan ke cli.py; alur training lengkap ada di train.py.
command = UV_RUN + [
    "run", "--stage", STAGE, "--student", STUDENT, "--method", METHOD,
    "--seed", str(SEED), "--mode", MODE, "--device", str(DEVICE),
    "--output-root", OUTPUT_ROOT, "--wandb-mode", WANDB_MODE,
]
if data_root:
    command += ["--dataset-root", str(data_root)]
if TEACHER_CKPT:
    command += ["--teacher-ckpt", TEACHER_CKPT]
if MODE == "full" and MODEL_WEIGHTS:
    command += ["--weights", str(MODEL_WEIGHTS)]
subprocess.run(command, cwd=REPO, env=CHILD_ENV, check=True)

## 5. Lihat dan simpan output

Setiap run memiliki direktori unik. `best.pt`, metrics, resolved config, metadata, `results.csv`, dan audit tersimpan di Working. Gunakan **Save Version** agar output sesi tersimpan; teacher antarsesi dapat dilampirkan kembali sebagai Kaggle Input atau diambil dari W&B artifact.

W&B upload per-epoch checkpoint default mati. Tidak ada full training atau multiseed yang diluncurkan otomatis.

In [ ]:
from IPython.display import FileLink, display

# Pointer menunjuk run terakhir; setiap run tetap punya folder hasil sendiri.
pointer = Path(OUTPUT_ROOT) / "latest_run.json"
latest = json.loads(pointer.read_text(encoding="utf-8"))
run_dir = Path(latest["run_dir"])
summary_path = run_dir / "run_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.is_file() else latest
metrics = summary.get("metrics", {})
compact = {key: summary.get(key) for key in ("status", "run_dir", "best_pt", "mode", "nonfinal", "best_epoch", "test_evaluated")}
compact["metrics"] = {key: metrics.get(key) for key in ("mAP50_95", "mAP50", "precision", "recall", "per_class")} if isinstance(metrics, dict) else metrics
print(json.dumps(compact, indent=2))
for filename in ("weights/best.pt", "metrics.json", "per_class.csv", "results.csv", "resolved_config.yaml", "metadata.json", "run_summary.json"):
    path = run_dir / filename
    if path.is_file():
        display(FileLink(str(path)))
if PLATFORM == "colab":
    print("Unduh/salin output ke Drive sebelum sesi Colab berakhir, atau simpan W&B artifact.")
else:
    print("Simpan output dengan Kaggle Save Version atau W&B artifact.")
# Log bootstrap juga tersedia pada output Kaggle/Colab.
for path in sorted(LOG_DIR.glob("*.log")):
    display(FileLink(str(path)))
